In [1]:
import torch
import numpy as np
from datasets import Dataset
from sentence_transformers import CrossEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm
import os

In [ ]:
from tablevault import tablevault

vault = tablevault.Vault(user_id="jinjin",
                            process_name="sentence_transformers_cross_encoder_mrpc",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=False,               
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()

In [ ]:
def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding

In [2]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

model_name = "cross-encoder/quora-distilroberta-base"
model = CrossEncoder(model_name, device=device)

print("model:", model_name)
print("num_labels:", getattr(model.config, "num_labels", None))
print("id2label:", getattr(model.config, "id2label", None))


device: mps


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cross-encoder/quora-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model: cross-encoder/quora-distilroberta-base
num_labels: 1
id2label: {0: 'LABEL_0'}


In [3]:
ds = vault.query_item_content("glue_mrpc_validation")
ds = Dataset.from_dict(ds)
print(ds)
print(ds[0])

sentence1 = ds["sentence1"]
sentence2 = ds["sentence2"]
pairs = list(zip(sentence1, sentence2))
y_true = np.array(ds["label"], dtype=int)

print("num_examples:", len(ds))
print("positive_rate:", float(y_true.mean()))


Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647


In [5]:
batch_size = 64
raw_scores = model.predict(pairs, batch_size=batch_size, show_progress_bar=True)
raw_scores = np.asarray(raw_scores)

print("raw_scores.shape:", raw_scores.shape)

if raw_scores.ndim == 2 and raw_scores.shape[1] == 2:
    duplicate_scores = raw_scores[:, 1]
    y_pred = np.argmax(raw_scores, axis=1).astype(int)
else:
    duplicate_scores = raw_scores.reshape(-1)
    y_pred = (duplicate_scores >= 0.5).astype(int)

print("done")


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

raw_scores.shape: (408,)
done


In [ ]:
vault.create_record_list("cross_encoder_classification", column_names=["prediction"])

for i in range(len(y_pred)):
    vault.append_record("cross_encoder_classification", {"prediction": y_pred[i]}, 
                       input_items = {"glue_mrpc_validation": [i, i + 1]}
                       )

description = "INSERT TEXT HERE ABOUT cross_encoder_classification"
embedding = get_embeddings(description)
vault.create_description("cross_encoder_classification", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("cross_encoder_classification", cat, embedding, prop)

In [5]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"])
print({"accuracy": acc, "f1": f1})
print(report)


{'accuracy': 0.6838235294117647, 'f1': 0.7667269439421338}
                precision    recall  f1-score   support

not_paraphrase       0.50      0.52      0.51       129
    paraphrase       0.77      0.76      0.77       279

      accuracy                           0.68       408
     macro avg       0.64      0.64      0.64       408
  weighted avg       0.69      0.68      0.69       408



In [6]:
for i in range(5):
    print("=" * 80)
    print("sentence1:", sentence1[i])
    print("sentence2:", sentence2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]), "score:", float(duplicate_scores[i]))


sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
true: 1 pred: 1 score: 0.9779019355773926
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
true: 0 pred: 0 score: 0.006291278637945652
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
true: 0 pred: 0 score: 0.04939701780676842
sentence1: The AFL-CIO is waiting until October to decide if it will endorse a candidate .
sentence2: The AFL-CIO announced Wednesday that it will 

In [7]:
mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

vault.create_record_list("sentence_transformers_cross_encoder_mrpc_mistakes", column_names=["idx", "sentence1", "sentence2", "true", "pred"])

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sentence1[i])
    print("sentence2:", sentence2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))

    vault.append_record("sentence_transformers_cross_encoder_mrpc_mistakes", {
            "idx": i,
            "sentence1": sentence1[i],
            "sentence2": sentence2[i],
            "true": int(y_true[i]),
            "pred": int(y_pred[i]),
        },
        input_items = {
            "glue_mrpc_validation": [i, i+ 1],
            "cross_encoder_classification": [i, i+ 1]
        }
    )

description = "INSERT TEXT HERE ABOUT sentence_transformers_cross_encoder_mrpc_mistakes"
embedding = get_embeddings(description)
vault.create_description("sentence_transformers_cross_encoder_mrpc_mistakes", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("sentence_transformers_cross_encoder_mrpc_mistakes", cat, embedding, prop)

num_errors: 129
idx: 5
sentence1: Wal-Mart said it would check all of its million-plus domestic workers to ensure they were legally employed .
sentence2: It has also said it would review all of its domestic employees more than 1 million to ensure they have legal status .
true: 1 pred: 0 score: 0.015424183569848537
idx: 6
sentence1: While dioxin levels in the environment were up last year , they have dropped by 75 percent since the 1970s , said Caswell .
sentence2: The Institute said dioxin levels in the environment have fallen by as much as 76 percent since the 1970s .
true: 0 pred: 1 score: 0.8578875660896301
idx: 7
sentence1: This integrates with Rational PurifyPlus and allows developers to work in supported versions of Java , Visual C # and Visual Basic .NET.
sentence2: IBM said the Rational products were also integrated with Rational PurifyPlus , which allows developers to work in Java , Visual C # and VisualBasic .Net.
true: 1 pred: 0 score: 0.25439533591270447
idx: 11
sentence1: 

In [8]:
vault.create_record_list("sentence_transformers_cross_encoder_mrpc_summary", column_names=["accuracy", "f1", "classification_report"])


summary = {
    "accuracy": float(acc),
    "f1": float(f1),
    "classification_report": str(report)
}

vault.append_record("sentence_transformers_cross_encoder_mrpc_summary", summary,
                    input_items = {
                        "glue_mrpc_validation": [0, len(ds)],
                        "cross_encoder_classification": [0, len(ds)]
                    })

summary

description = "INSERT TEXT HERE ABOUT sentence_transformers_cross_encoder_mrpc_summary"
embedding = get_embeddings(description)
vault.create_description("sentence_transformers_cross_encoder_mrpc_summary", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("sentence_transformers_cross_encoder_mrpc_summary", cat, embedding, prop)

{'dataset': 'glue/mrpc',
 'split': 'validation',
 'model': 'cross-encoder/quora-distilroberta-base',
 'device': 'mps',
 'num_examples': 408,
 'accuracy': 0.6838235294117647,
 'f1': 0.7667269439421338}

In [ ]:
description = "INSERT TEXT HERE ABOUT sentence_transformers_cross_encoder_mrpc process" # description of whole notebook
embedding = get_embeddings(description)
vault.create_description("sentence_transformers_cross_encoder_mrpc", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. model: distilbert-base-uncased-MRPC

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("sentence_transformers_cross_encoder_mrpc", cat, embedding, prop)